# Clinical Reasoning Agent — retrieval A/B

Measures what retrieval changes, by running the same five scenarios twice: once with
`retrieved=None` (reproducing the frozen pre-RAG baseline) and once with TF-IDF retrieval
over the 30-unit corpus.

**What this can establish.** Whether adding retrieval changes the validators' verdicts on
five designed cases, and in which direction.

**What it cannot.** Anything about diagnostic accuracy. There is no clinical ground truth
here. The cases are synthetic and the comparison is against the system's own safety layer,
not against what a clinician would have concluded.

**The check that matters most** is in section 9: the no-retrieval arm must reproduce the
frozen baseline. If it does not, something other than retrieval changed and the whole
comparison is confounded.

**Known limitation, recorded before running.** Unit F20 — what a negative FAST does not
exclude — does not retrieve for its own question. TF-IDF ranks F18 above it because F18
repeats "FAST negative" in a short passage while F20 is long and says "false-negative" and
"sensitivity" instead. If a case fails to improve, this is a candidate explanation.

Runtime: **Runtime → Change runtime type → T4 GPU**. About 15 minutes end to end.

## 1. Runtime

In [4]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU"

Mon Aug 17 16:47:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. llama-cpp-python (CUDA build)

In [5]:
# Prebuilt CUDA wheel. If this fails, use the fallback cell below.
!pip -q install llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

import llama_cpp
print("llama_cpp", llama_cpp.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 GB 961.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.1 MB/s eta 0:00:00
llama_cpp 0.3.35


In [6]:
# FALLBACK ONLY -- run this only if the cell above failed. Compiles from source, ~10 min.
# !CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --force-reinstall --no-cache-dir

## 3. Model

In [7]:
!pip -q install huggingface_hub
from pathlib import Path
from huggingface_hub import hf_hub_download

# Downloaded inside Colab rather than uploaded: ~2 minutes on Google's network against
# hours to push 4.6 GB from a home connection. Lands in /content, cleared on runtime
# restart, so this re-runs each session.
MODEL = Path(hf_hub_download(
    repo_id='bartowski/HuatuoGPT-o1-8B-GGUF',
    filename='HuatuoGPT-o1-8B-Q4_K_M.gguf',
    local_dir='/content/model'))

print(f"ready: {MODEL}  ({MODEL.stat().st_size/1e9:.1f} GB)")

HuatuoGPT-o1-8B-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B / 4.92GB            

HuatuoGPT-o1-8B-Q4_K_M.gguf: downloading bytes:           |  0.00B            

ready: /content/model/HuatuoGPT-o1-8B-Q4_K_M.gguf  (4.9 GB)


## 4. Code

Upload **`pocus_agents_colab.zip`**. It must be the rebuilt one — it carries corpus v1.0
(all 30 units sourced) and the frozen baseline used by section 9.

In [8]:
from google.colab import files
import zipfile, sys, os

up = files.upload()                       # pick pocus_agents_colab.zip
with zipfile.ZipFile(next(iter(up))) as z:
    z.extractall('/content/pocus')

# Scrub the import path before importing anything. A copy of `src` in Drive takes priority
# otherwise, and Python keeps serving whichever version it imported first -- which once
# produced byte-identical results from "new" code and looked like a finding rather than a
# mistake.
sys.path = [p for p in sys.path if 'POCUS-Project' not in p and 'drive' not in p]
sys.path.insert(0, '/content/pocus')
for m in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.')]:
    del sys.modules[m]

os.environ['POCUS_LLM_PATH'] = str(MODEL)

import src.agents.clinical.reasoning as R
print('loaded from  :', R.__file__)
# Reads the COMPILED function, so a newer file sitting unused on disk cannot fool it.
print('NEW code live:', 'focus' in R.reason.__code__.co_varnames)

Saving pocus_agents_colab.zip to pocus_agents_colab.zip
loaded from  : /content/pocus/src/agents/clinical/reasoning.py
NEW code live: True


## 5. Corpus and retrieval sanity

Costs nothing and runs before the GPU work. Two things it catches: a stale zip, and a
scenario that retrieves nothing — whose two arms would then be identical by construction.

In [9]:
import json
from src.agents.clinical.retrieval import Retriever, retrieval_note, is_grounded
from src.agents.clinical.run_case import SCENARIOS, build

corpus = json.load(open('/content/pocus/src/agents/clinical/corpus/pocus_corpus.json'))
print('corpus version :', corpus['corpus_version'])
print('sourced        :', sum(p['status'] == 'sourced' for p in corpus['passages']),
      '/', len(corpus['passages']))
assert corpus['corpus_version'] == '1.0-all-units-sourced', 'stale zip -- re-upload'

retriever = Retriever()
for s in SCENARIOS:
    hits = retriever.for_state(build(s))
    print(f"\n{s}")
    print("   ", retrieval_note(hits), "| grounded:", is_grounded(hits))
    for h in hits:
        print(f"     [{h['n']}] {h['id']} {h['topic']:<28} {h['score']:.3f}")

corpus version : 1.0-all-units-sourced
sourced        : 30 / 30

missing
    4 sourced passage(s) retrieved | grounded: True
     [1] L07 consolidation                0.208
     [2] B26 acute_dyspnoea_approach      0.173
     [3] L08 pleural_effusion             0.144
     [4] L05 lung_sliding_absent          0.124

conflict
    1 sourced passage(s) retrieved | grounded: True
     [1] C12 rv_dilation                  0.101

concordant
    4 sourced passage(s) retrieved | grounded: True
     [1] L07 consolidation                0.213
     [2] B26 acute_dyspnoea_approach      0.160
     [3] L08 pleural_effusion             0.148
     [4] L05 lung_sliding_absent          0.127

reassuring
    4 sourced passage(s) retrieved | grounded: True
     [1] L07 consolidation                0.232
     [2] B26 acute_dyspnoea_approach      0.174
     [3] L08 pleural_effusion             0.160
     [4] L10 blue_protocol                0.111

not_assessed
    4 sourced passage(s) retrieved | grounded: 

## 6. Safety benchmark — the layer under test, before the model is involved

In [10]:
!cd /content/pocus && python -m src.agents.tests.run_benchmark

CLINICAL REASONING AGENT -- SAFETY BENCHMARK
Safety property               Tests  Passed
--------------------------------------------------
Absent is not normal             17      17
Advice scope                      4       4
Benchmark scenarios               9       9
Case-quality grading              6       6
Confidence calibration            9       9
Conflict detection                9       9
Enumerated evidence               8       8
Escalation policy                11      11
Evidence coverage                 8       8
Evidence relationships            5       5
Failure severity                  3       3
Hallucination rejection          25      25
LLM failure containment           5       5
Malformed output rejection       15      15
Model-scope propagation           5       5
Reference-range detection         5       5
Retrieval grounding              19      19
Unassessed-organ reporting        5       5
Value-reading consistency         3       3
------------------------

## 7. Load the model

In [11]:
import time
from src.agents.clinical.llm import LlamaCppBackend

t0 = time.time()
backend = LlamaCppBackend(n_gpu_layers=-1, n_ctx=4096, max_tokens=1400, verbose=False)
print(f"model loaded in {time.time()-t0:.1f}s")

model loaded in 17.3s


## 8. The A/B run

Arm A (`none`) runs entirely before arm B (`tfidf`), so arm A can be checked against the
frozen baseline as a block rather than interleaved with runs that changed the prompt.

`max_revisions=1`: a revision request carries one complaint, so one round can fix at most
one unsound fault.

In [12]:
import time, json
from src.agents.clinical.run_case import SCENARIOS, build
from src.agents.clinical.reasoning import reason
from src.agents.clinical.retrieval import Retriever, retrieval_note, is_grounded

retriever = Retriever()
ab = {}

for arm, use_retrieval in (('none', False), ('tfidf', True)):
    ab[arm] = {}
    print(f"\n{'='*70}\nARM: {arm}\n{'='*70}")
    for s in SCENARIOS:
        st = build(s)
        hits = retriever.for_state(st) if use_retrieval else None
        t0 = time.time()
        out = reason(st, llm_fn=backend, retrieved=hits, max_revisions=1)
        dt = time.time() - t0
        out['_seconds'] = dt
        out['_retrieval'] = {
            'passages': len(hits or []),
            'ids': [h['id'] for h in (hits or [])],
            'grounded': is_grounded(hits or []),
            'note': retrieval_note(hits or []),
        }
        ab[arm][s] = out
        print(f"  {s:<14} {dt:6.1f}s  "
              f"withheld={str(out.get('differential_withheld', False)):<5} "
              f"errors={len(out['validation_errors'] or [])}  "
              f"warnings={len(out.get('warnings') or [])}  "
              f"revisions={len(out.get('revisions') or [])}  "
              f"hits={out['_retrieval']['ids']}")

json.dump(ab, open('/content/ab_retrieval.json', 'w'), indent=2, default=str)
print('\nsaved /content/ab_retrieval.json')


ARM: none
  missing          47.2s  withheld=False errors=0  warnings=2  revisions=1  hits=[]
  conflict         53.7s  withheld=False errors=0  warnings=1  revisions=1  hits=[]
  concordant       83.8s  withheld=False errors=0  warnings=2  revisions=1  hits=[]
  reassuring       36.6s  withheld=False errors=0  warnings=0  revisions=0  hits=[]
  not_assessed     68.6s  withheld=False errors=0  warnings=2  revisions=1  hits=[]

ARM: tfidf
  missing          66.9s  withheld=False errors=0  warnings=2  revisions=1  hits=['L07', 'B26', 'L08', 'L05']
  conflict         33.0s  withheld=False errors=0  warnings=0  revisions=0  hits=['C12']
  concordant       75.9s  withheld=False errors=0  warnings=2  revisions=1  hits=['L07', 'B26', 'L08', 'L05']
  reassuring       66.0s  withheld=False errors=0  warnings=0  revisions=1  hits=['L07', 'B26', 'L08', 'L10']
  not_assessed     70.3s  withheld=True  errors=1  warnings=3  revisions=1  hits=['L07', 'B26', 'L08', 'L05']

saved /content/ab_retrieval

## 9. Comparison

The first block is the one that can invalidate everything else.

In [15]:
import json
base = json.load(open('/content/pocus/models/clinical_reasoning_pre_rag/results.json'))

def errs(d):
    return d['validation_errors'] or []

print("Does the no-retrieval arm reproduce the frozen baseline?")
print("(NOTE: enumerated evidence changed the prompt, so DIFFERS is expected here --")
print(" the pre-RAG baseline was frozen against the old free-text interface)\n")
for s in ab['none']:
    same_diff = ab['none'][s]['differential'] == base[s]['differential']
    same_esc  = ab['none'][s]['escalation']   == base[s]['escalation']
    print(f"  {s:<14} differential={'MATCH' if same_diff else 'DIFFERS'}   "
          f"escalation={'MATCH' if same_esc else 'DIFFERS'}")

print("\n\nEscalation must be identical across arms -- computed before the model runs.")
for s in ab['none']:
    a, b = ab['none'][s]['escalation'], ab['tfidf'][s]['escalation']
    print(f"  {s:<14} {'OK' if a == b else 'VIOLATED -- investigate'}")

print(f"\n\n{'scenario':<16}{'withheld':<14}{'errors':<12}{'warnings':<14}grounded")
print('-' * 72)
for s in ab['none']:
    a, b = ab['none'][s], ab['tfidf'][s]
    wa, wb = a.get('differential_withheld', False), b.get('differential_withheld', False)
    print(f"  {s:<14}{str(wa)[0]} -> {str(wb)[0]:<9}"
          f"{len(errs(a))} -> {len(errs(b)):<8}"
          f"{len(a.get('warnings') or [])} -> {len(b.get('warnings') or []):<10}"
          f"{b['_retrieval']['grounded']}")

print("\n\nAll errors, no-retrieval arm:")
for s in ab['none']:
    for e in errs(ab['none'][s]):
        print(f"  [{s}] {e}")

print("\nAll errors, tfidf arm:")
for s in ab['tfidf']:
    for e in errs(ab['tfidf'][s]):
        print(f"  [{s}] {e}")

Does the no-retrieval arm reproduce the frozen baseline?
(NOTE: enumerated evidence changed the prompt, so DIFFERS is expected here --
 the pre-RAG baseline was frozen against the old free-text interface)

  missing        differential=DIFFERS   escalation=MATCH
  conflict       differential=DIFFERS   escalation=MATCH
  concordant     differential=DIFFERS   escalation=MATCH
  reassuring     differential=DIFFERS   escalation=MATCH
  not_assessed   differential=DIFFERS   escalation=MATCH


Escalation must be identical across arms -- computed before the model runs.
  missing        OK
  conflict       OK
  concordant     OK
  reassuring     OK
  not_assessed   OK


scenario        withheld      errors      warnings      grounded
------------------------------------------------------------------------
  missing       F -> F        0 -> 0       2 -> 2         True
  conflict      F -> F        0 -> 0       1 -> 0         True
  concordant    F -> F        0 -> 0       2 -> 2         True
  

## 10. Reproducibility

Temperature 0, fixed seed, KV cache reset per call. If two identical calls diverge, every
difference measured above is noise.

In [16]:
st = build('missing')
hits = retriever.for_state(st)
a = reason(st, llm_fn=backend, retrieved=hits, max_revisions=1)
b = reason(st, llm_fn=backend, retrieved=hits, max_revisions=1)
print('identical across two calls:', a['differential'] == b['differential'])

identical across two calls: True


## 11. Download

In [18]:
from google.colab import files
files.download('/content/ab_retrieval.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>